# The People's Common Sense Medical Adviser — Qwen3.5 Vision OCR

Page-faithful OCR of a scanned 19th-century English medical book, plus a **separate**
figure-description corpus, for a citable RAG index.

Adapted from the Bengali MP3 notebook. What changed and why:

| Change | Reason |
|---|---|
| Every page is transcribed; none are skipped | The Bengali run dropped front matter. Here the degraded preface pages are EDA evidence, and page/word counts must be honest. |
| The second "structure" LLM call is gone | It was ~half the runtime and was where restructuring hallucinations entered. English prose chunks deterministically in Python. |
| Figures are cropped **geometrically** (PyMuPDF), then described one crop at a time | A small engraving inside a 3200px page gets a handful of visual tokens and a vague description. A native-resolution crop is sharp, fast and cheap. |
| The **printed legend** is captured verbatim, separately from the model's description | This book prints real legends ("Fig. 8. Thigh-bone, sawn open lengthwise."). That is author-written ground truth for the image — it makes `legend_agreement` a free hallucination detector over ~1000 figures. |
| `pdf_page` and `printed_page` are both stored | PDF index != printed page number. Citing the index makes every human-facing citation wrong by a constant offset. |
| `clean_text` (faithful) and `normalized_text` (de-hyphenated, ligatures expanded) | Faithful text is what you score OCR against; normalized text is what you embed. |
| Optional confidence routing via `PAGE_QUALITY_CSV` | Lets the later Tesseract pass decide which pages actually need the GPU. |

**Outputs**

- `pcma_page_ocr.jsonl` — one record per page: faithful text, page numbers, headings, quality flags, raw model output, usage.
- `pcma_figures.jsonl` — one record per figure: crop path, printed legend, model description, legend agreement. Stored separately, per the supervisor's requirement.
- `pcma_chunks.jsonl` — deterministic text chunks with page provenance.
- `pcma_rag_corpus.jsonl` / `.csv` — text chunks + figure records in one file, tagged `modality`, ready for a single vector index.
- `pcma_pages.csv`, `pcma_token_log.csv`, `pcma_errors.csv`, `pcma_rag_readable.txt`.

Do not delete `pcma_page_ocr.jsonl` after indexing. It is the audit trail.


## 1. Configuration


In [ ]:
import base64
import contextlib
import gc
import json
import os
import re
import statistics
import subprocess
import sys
import time
import unicodedata
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageOps

KAGGLE_INPUT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
WORK_DIR.mkdir(parents=True, exist_ok=True)

BOOK_SLUG = "pcma"
BOOK_TITLE = "The People's Common Sense Medical Adviser (R. V. Pierce)"

# ---------------------------------------------------------------- input PDFs
# Either the single full-book PDF, or the folder of 50-page chunks.
# Chunk filenames of the form "..._pages_0251-0300.pdf" are detected and their
# page numbers are mapped back to absolute book pages automatically.
PDF_ROOT_PATH = "/kaggle/input/pcma-book"
PDF_PATH = None
PROCESS_ALL_PDFS_IN_ROOT = True
PDF_NAME_CONTAINS = None  # e.g. "common-sense"; case-insensitive, or None

# ---------------------------------------------------------------- model files
MODEL_PATH = Path("/kaggle/input/datasets/leqvinh/qwen3-5-9b-ud-q8-k-xl-gguf/Qwen3.5-9B-UD-Q8_K_XL.gguf")
MMPROJ_PATH = Path("/kaggle/input/datasets/kmazd1110/qwen3-5-vision-mmproj/mmproj_assets/models/mmproj-F16.gguf")

MODEL_CANDIDATE_NAMES = ["Qwen3.5-9B-UD-Q8_K_XL.gguf", "Qwen3.5-9B-BF16.gguf"]
MMPROJ_CANDIDATE_NAMES = ["mmproj-F16.gguf", "mmproj-BF16.gguf", "mmproj-F32.gguf"]

LLAMA_CPP_WHEEL_DIRS = [
    Path("/kaggle/input/datasets/inf3cted/offline-wheel-iut/offline_assets/wheels"),
]
PYMUPDF_WHEEL_DIRS = [
    Path("/kaggle/input/datasets/kmazd1110/py-mu-pdf-wheel/offline_assets/wheels"),
]
INSTALL_FROM_OFFLINE_WHEELS = True

# ---------------------------------------------------------------- page range
# The measured Bengali rate was 60.6 s/page for two calls. One call plus figures
# should land well under that, but 1034 pages will still not fit one session at
# full-page VLM. Split the book across sessions with START_PAGE / END_PAGE, or
# route with PAGE_QUALITY_CSV below. Resume is keyed on (pdf, page) either way.
START_PAGE = 1
END_PAGE = None      # inclusive absolute book page, or None for "to the end"
PAGE_LIMIT = 5       # SMOKE TEST. Set to None for a full run.

# ---------------------------------------------------------------- rendering
# Section 4 prints the native scan DPI. Do not render above it: upscaling a
# 200 DPI scan to 400 DPI buys blur and doubles the image tokens.
RENDER_DPI = 300
IMAGE_MAX_SIDE = 2200     # page image handed to the VLM
FIGURE_MAX_SIDE = 1400    # figure crop handed to the VLM

RESET_OUTPUTS = False
RESUME = True
RETRY_ERRORS = True
SAVE_EVERY = 10

# ---------------------------------------------------------------- figures
EXTRACT_FIGURES = True     # geometric crop; cheap, no GPU, no hallucination
DESCRIBE_FIGURES = True    # one VLM call per crop
FIGURE_DETECTION_MODE = "auto"   # "auto" | "embedded" | "textgap"

FIG_MIN_WIDTH_FRAC = 0.10   # of page width
FIG_MIN_HEIGHT_FRAC = 0.04  # of page height
FIG_MAX_AREA_FRAC = 0.70    # drops the full-page scan/MRC layers
FIG_MIN_INK_FRAC = 0.015    # drops blank whitespace bands
FIG_MIN_GAP_PTS = 34.0      # textgap mode: smallest vertical gap that can be a figure
TEXT_LINE_MERGE_PTS = 6.0   # textgap mode: gap below this is normal leading
FIG_LEGEND_LOOKAHEAD_PTS = 130.0
FIG_LABEL_LOOKBACK_PTS = 40.0
FIG_PAD_PTS = 5.0
INK_THRESHOLD = 205         # grey level below which a pixel counts as ink
MAX_FIGURES_PER_PAGE = 8

# ---------------------------------------------------------------- routing
# Optional. CSV with columns: source_pdf, page, ocr_quality (0..1) from the
# Tesseract pass. When set, only pages at or below the threshold go to the VLM.
PAGE_QUALITY_CSV = None
PAGE_QUALITY_THRESHOLD = 0.80

# ---------------------------------------------------------------- chunking
# Deterministic Python, not a model call.
CHUNK_TARGET_CHARS = 900
CHUNK_MAX_CHARS = 1400
CHUNK_OVERLAP_CHARS = 150
CHUNK_MIN_CHARS = 120
CHUNKABLE_PAGE_TYPES = {"prose", "mixed", "table", "figure_plate"}

# ---------------------------------------------------------------- llama.cpp
N_CTX = 16384
N_GPU_LAYERS = -1
N_BATCH = 512
N_THREADS = max(2, os.cpu_count() or 2)
MAX_OCR_TOKENS = 4096
MAX_FIGURE_TOKENS = 768
TEMPERATURE = 0.0
TOP_P = 1.0

# ---------------------------------------------------------------- outputs
PAGE_OCR_JSONL = WORK_DIR / f"{BOOK_SLUG}_page_ocr.jsonl"
FIGURES_JSONL = WORK_DIR / f"{BOOK_SLUG}_figures.jsonl"
CHUNKS_JSONL = WORK_DIR / f"{BOOK_SLUG}_chunks.jsonl"
RAG_CORPUS_JSONL = WORK_DIR / f"{BOOK_SLUG}_rag_corpus.jsonl"
RAG_CORPUS_CSV = WORK_DIR / f"{BOOK_SLUG}_rag_corpus.csv"
PAGES_CSV = WORK_DIR / f"{BOOK_SLUG}_pages.csv"
FIGURES_CSV = WORK_DIR / f"{BOOK_SLUG}_figures.csv"
TOKEN_LOG_CSV = WORK_DIR / f"{BOOK_SLUG}_token_log.csv"
ERROR_LOG_CSV = WORK_DIR / f"{BOOK_SLUG}_errors.csv"
READABLE_TXT = WORK_DIR / f"{BOOK_SLUG}_rag_readable.txt"

IMAGE_CACHE_DIR = WORK_DIR / f"{BOOK_SLUG}_page_images"
FIGURE_DIR = WORK_DIR / f"{BOOK_SLUG}_figures"
IMAGE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Work directory:", WORK_DIR)
print("Smoke test:", "yes, PAGE_LIMIT=%s" % PAGE_LIMIT if PAGE_LIMIT else "no, full run")

# ---------------------------------------------------------------- chandra layout & ground truth
CHANDRA_PATH = Path("/kaggle/input/datasets/cruelangelssprint/pierce-1890-figure-and-ocr-outputs/chandra/chunks.jsonl")
LABELS_PATH = Path("/kaggle/input/datasets/kmazd1110/pierce-book-gt/labels.jsonl")
USE_CHANDRA_LAYOUT = True


## 2. Offline dependency setup


In [ ]:
def resolve_wheel_dir(candidate_dirs, label):
    for wheel_dir in candidate_dirs:
        if wheel_dir.exists() and list(wheel_dir.glob("*.whl")):
            return wheel_dir
    checked = "\n".join(f" - {p}" for p in candidate_dirs)
    raise FileNotFoundError(f"No wheelhouse found for {label}. Checked:\n{checked}")


def pip_install_offline(packages, wheel_dirs, label):
    if not INSTALL_FROM_OFFLINE_WHEELS:
        return
    wheel_dir = resolve_wheel_dir(wheel_dirs, label)
    cmd = [
        sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
        "--find-links", str(wheel_dir), *packages,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-3000:])
        raise RuntimeError(f"Offline install failed for {label}")


try:
    import pypdf
except Exception as exc:
    raise ImportError("pypdf is expected in the Kaggle runtime") from exc

try:
    import fitz
except Exception:
    pip_install_offline(["PyMuPDF"], PYMUPDF_WHEEL_DIRS, "PyMuPDF")
    import fitz

try:
    from llama_cpp import Llama
    import llama_cpp
except Exception:
    pip_install_offline(["llama-cpp-python"], LLAMA_CPP_WHEEL_DIRS, "llama-cpp-python")
    from llama_cpp import Llama
    import llama_cpp

print("pypdf:", getattr(pypdf, "__version__", "unknown"))
print("pymupdf:", getattr(fitz, "__doc__", "unknown"))
print("llama_cpp:", getattr(llama_cpp, "__version__", "unknown"))


## 3. Locate PDFs, model, and multimodal projector

Chunked scans are supported. A file named `..._pages_0251-0300.pdf` gets an offset of
250, so its internal page 7 is recorded as absolute book page 257. Every page number
in every output is an **absolute book page**, so a chunked run and a full-book run
produce interchangeable records.


In [ ]:
CHUNK_RANGE_RE = re.compile(r"pages?[_-](\d{1,5})\s*[-_to]{1,3}\s*(\d{1,5})", re.IGNORECASE)


def safe_slug(path):
    name = Path(path).stem.lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_") or "pdf"


def pdf_page_offset(pdf_file):
    """0 for a full book; start-1 for a '..._pages_0251-0300.pdf' chunk."""
    match = CHUNK_RANGE_RE.search(Path(pdf_file).name)
    return int(match.group(1)) - 1 if match else 0


def find_named_file(primary_path, candidate_names, suffix):
    if primary_path and Path(primary_path).exists():
        return Path(primary_path)
    for root in [KAGGLE_INPUT, Path("."), WORK_DIR]:
        if not root.exists():
            continue
        for name in candidate_names:
            matches = sorted(root.rglob(name))
            if matches:
                return matches[0]
    raise FileNotFoundError(f"Could not find {suffix}. Update the configuration cell.")


def discover_pdfs():
    if PDF_PATH is not None:
        pdf = Path(PDF_PATH)
        if not pdf.exists():
            raise FileNotFoundError(f"PDF_PATH does not exist: {pdf}")
        return [pdf]

    if PDF_ROOT_PATH is not None:
        root = Path(PDF_ROOT_PATH)
        if not root.exists():
            raise FileNotFoundError(f"PDF_ROOT_PATH does not exist: {root}")
        pdfs = sorted(root.rglob("*.pdf"))
    else:
        pdfs = sorted(KAGGLE_INPUT.rglob("*.pdf")) if KAGGLE_INPUT.exists() else sorted(Path(".").rglob("*.pdf"))

    if PDF_NAME_CONTAINS:
        needle = PDF_NAME_CONTAINS.lower()
        pdfs = [p for p in pdfs if needle in p.name.lower()]

    if not pdfs:
        raise FileNotFoundError("No PDFs found. Set PDF_ROOT_PATH or PDF_PATH.")

    if PDF_ROOT_PATH is None and PDF_PATH is None and len(pdfs) != 1:
        print("Visible PDFs:")
        for p in pdfs[:100]:
            print(" -", p)
        raise ValueError("Multiple PDFs found. Set PDF_ROOT_PATH, PDF_PATH, or PDF_NAME_CONTAINS.")

    return pdfs if PROCESS_ALL_PDFS_IN_ROOT else [pdfs[0]]


_DOC_CACHE = {}


def get_doc(pdf_file):
    """Open each PDF once. Reopening 1034 times is a measurable share of runtime."""
    key = str(pdf_file)
    if key not in _DOC_CACHE:
        _DOC_CACHE[key] = fitz.open(key)
    return _DOC_CACHE[key]


PDF_FILES = discover_pdfs()
MODEL_FILE = find_named_file(MODEL_PATH, MODEL_CANDIDATE_NAMES, ".gguf model")
MMPROJ_FILE = find_named_file(MMPROJ_PATH, MMPROJ_CANDIDATE_NAMES, "mmproj .gguf")

TOTAL_BOOK_PAGES = 0
print("PDFs to process:", len(PDF_FILES))
for pdf in PDF_FILES:
    n = get_doc(pdf).page_count
    offset = pdf_page_offset(pdf)
    TOTAL_BOOK_PAGES += n
    has_text = bool(get_doc(pdf).load_page(0).get_text("text").strip())
    print(f" - {pdf.name}: {n} pages | book pages {offset + 1}-{offset + n} | embedded text layer: {has_text}")
print("Total pages across inputs:", TOTAL_BOOK_PAGES)
print("Model:", MODEL_FILE)
print("MMProj:", MMPROJ_FILE)


## 4. Render pages, and check the native scan resolution first

`report_native_scan_dpi` is a two-line EDA finding you can quote in Section 3: it reports
the resolution the page was actually scanned at. Rendering above that number produces
upscaled blur, costs more image tokens, and makes Tesseract *worse*, not better.

Preprocessing is deliberately mild. Your own EDA already found that naive fixed-threshold
binarisation introduced new errors ("carminative" -> "earminative"), so there is no
binarisation here at all.


In [ ]:
def report_native_scan_dpi(pdf_file, page_number_in_file):
    """Native DPI of the largest embedded image on the page."""
    page = get_doc(pdf_file).load_page(page_number_in_file - 1)
    best = None
    for info in page.get_images(full=True):
        xref, width, height = info[0], info[2], info[3]
        rects = page.get_image_rects(xref) or []
        for rect in rects:
            if rect.width <= 1 or rect.height <= 1:
                continue
            dpi_x = width / (rect.width / 72.0)
            dpi_y = height / (rect.height / 72.0)
            area = abs(rect.get_area())
            if best is None or area > best["placed_area_pts2"]:
                best = {
                    "xref": xref, "pixels": f"{width}x{height}",
                    "native_dpi_x": round(dpi_x), "native_dpi_y": round(dpi_y),
                    "placed_area_pts2": round(area),
                }
    return best


def render_page(pdf_file, page_number_in_file, dpi=None):
    dpi = dpi or RENDER_DPI
    page = get_doc(pdf_file).load_page(page_number_in_file - 1)
    zoom = dpi / 72.0
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
    return Image.open(BytesIO(pix.tobytes("png"))).convert("RGB")


def preprocess_image(image):
    # No binarisation: fixed thresholding measurably hurt this corpus.
    image = ImageOps.exif_transpose(image).convert("RGB")
    image = ImageEnhance.Contrast(image).enhance(1.15)
    image = ImageEnhance.Sharpness(image).enhance(1.10)
    return image


def resize_max_side(image, max_side):
    if not max_side or max(image.size) <= max_side:
        return image
    scale = max_side / max(image.size)
    size = (max(1, int(image.width * scale)), max(1, int(image.height * scale)))
    resample = getattr(getattr(Image, "Resampling", Image), "LANCZOS")
    return image.resize(size, resample)


def page_image(pdf_file, page_number_in_file, book_page):
    page_dir = IMAGE_CACHE_DIR / safe_slug(pdf_file)
    page_dir.mkdir(parents=True, exist_ok=True)
    image = resize_max_side(preprocess_image(render_page(pdf_file, page_number_in_file)), IMAGE_MAX_SIDE)
    path = page_dir / f"page_{book_page:04d}.png"
    image.save(path, optimize=True)
    return path, image


_probe_pdf = PDF_FILES[0]
_probe_local = max(1, START_PAGE - pdf_page_offset(_probe_pdf))
_probe_local = min(_probe_local, get_doc(_probe_pdf).page_count)
print("Native scan resolution of", _probe_pdf.name, "page", _probe_local, "->", report_native_scan_dpi(_probe_pdf, _probe_local))
print("RENDER_DPI is set to", RENDER_DPI, "- lower it if it exceeds the native DPI above.\n")

_preview_path, _preview_image = page_image(_probe_pdf, _probe_local, pdf_page_offset(_probe_pdf) + _probe_local)
print("Preview:", _preview_path, _preview_image.size)
display(resize_max_side(_preview_image, 700))


# ---------------------------------------------------------------- Chandra Layout Helpers
def _chandra_label_kind(label) -> str:
    TEXT_LABELS = {
        "text", "section-header", "caption",
        "footnote", "list-group", "table",
    }
    if label is None:
        return "skip"
    return "text" if str(label).lower().strip() in TEXT_LABELS else "skip"


def load_chandra_blocks(chandra_path):
    if not chandra_path or not Path(chandra_path).exists():
        print(f"[WARN] Chandra chunks.jsonl not found at {chandra_path}")
        return {}
    blocks_by_page = defaultdict(list)
    with Path(chandra_path).open(encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            book_page = row.get("book_page")
            if book_page is None:
                continue
            label = row.get("label", "")
            if _chandra_label_kind(label) == "skip":
                continue
            page_id = f"p{int(book_page):04d}"
            blocks_by_page[page_id].append({
                "page_box": row.get("page_box"),
                "bbox": row.get("bbox"),
                "label": label,
                "content": re.sub(r"<[^>]+>", " ", row.get("content", "")).strip(),
            })
    print(f"Loaded Chandra layout: {sum(len(v) for v in blocks_by_page.values())} text blocks across {len(blocks_by_page)} pages.")
    return blocks_by_page


def bbox_to_pixel(bbox, page_box, img_w, img_h):
    if not bbox or not page_box or len(page_box) < 4 or len(bbox) < 4:
        return None
    pb_x0, pb_y0, pb_x1, pb_y1 = (float(v) for v in page_box)
    cw = pb_x1 - pb_x0
    ch = pb_y1 - pb_y0
    if cw <= 0.0 or ch <= 0.0:
        return None
    x0, y0, x1, y1 = (float(v) for v in bbox)
    px0 = max(0, int((x0 - pb_x0) / cw * img_w))
    py0 = max(0, int((y0 - pb_y0) / ch * img_h))
    px1 = min(img_w, int((x1 - pb_x0) / cw * img_w))
    py1 = min(img_h, int((y1 - pb_y0) / ch * img_h))
    return px0, py0, px1, py1


## 5. Page furniture: printed page number, running head, historical text normalisation

The recto running head names the current section on nearly every page
(`GENERATION.`, `THE BONES.`). Stripped from the body it is noise; kept as metadata it
gives you chapter structure across 1034 pages for free — which is the atomic unit your
no-leakage split promises.

`normalize_historical_text` expands `æ/œ/ﬁ`, maps long-s to `s`, and joins words the
printer broke across lines. `clean_text` stays faithful for OCR scoring; `normalized_text`
is what gets embedded.


In [ ]:
LIGATURES = {
    "ﬀ": "ff", "ﬁ": "fi", "ﬂ": "fl", "ﬃ": "ffi", "ﬄ": "ffl",
    "ſ": "s",   # long s
    "æ": "ae", "Æ": "AE", "œ": "oe", "Œ": "OE",
    "‘": "'", "’": "'", "“": '"', "”": '"',
    "–": "-", "—": "-", "‐": "-", "‑": "-",
}

TERMINAL_PUNCT = tuple(".!?:;\"')]")


def clean_string(value):
    return re.sub(r"[ \t]+", " ", str(value or "")).strip()


def normalize_historical_text(text):
    """Retrieval-facing normalisation. Never overwrite clean_text with this."""
    out = unicodedata.normalize("NFKC", str(text or ""))
    for src, dst in LIGATURES.items():
        out = out.replace(src, dst)
    # Join printer hyphenation across a line break: "carmi-\nnative" -> "carminative".
    out = re.sub(r"([A-Za-z])[-­]\n[ \t]*([a-z])", r"\1\2", out)
    # Soft-wrap the remaining single newlines inside a paragraph.
    out = re.sub(r"(?<!\n)\n(?!\n)", " ", out)
    out = re.sub(r"[ \t]{2,}", " ", out)
    out = re.sub(r"\n{3,}", "\n\n", out)
    return out.strip()


def page_furniture_from_textlayer(page):
    """Printed page number and running head, read from the PDF's own text layer."""
    rect = page.rect
    top_band = rect.height * 0.11
    bottom_band = rect.height * 0.89
    top_text, bottom_text = [], []
    for block in page.get_text("blocks"):
        if block[6] != 0:
            continue
        text = str(block[4]).strip()
        if not text:
            continue
        y0, y1 = block[1], block[3]
        if y0 < top_band:
            top_text.append(text)
        elif y1 > bottom_band:
            bottom_text.append(text)

    printed_page = ""
    for source in (top_text, bottom_text):
        joined = " ".join(source)
        numbers = re.findall(r"\b(\d{1,4})\b", joined)
        if numbers:
            printed_page = numbers[0] if len(numbers) == 1 else max(numbers, key=len)
            break

    head = " ".join(top_text)
    head = re.sub(r"\b\d{1,4}\b", " ", head)
    head = clean_string(head)
    return {"printed_page_textlayer": printed_page, "running_head_textlayer": head}


def reconcile_printed_pages(rows):
    """PDF index != printed page. Derive the offset from agreement, then fill gaps.

    rows: list of dicts with pdf_page, printed_page_textlayer, printed_page_vlm.
    Mutates each row to set printed_page and printed_page_source.
    """
    offsets = []
    for row in rows:
        for key in ("printed_page_textlayer", "printed_page_vlm"):
            value = str(row.get(key) or "").strip()
            if value.isdigit():
                offsets.append(int(value) - int(row["pdf_page"]))
    modal_offset = statistics.mode(offsets) if offsets else 0

    for row in rows:
        pdf_page = int(row["pdf_page"])
        expected = pdf_page + modal_offset
        chosen, source = None, "offset"
        for key, label in (("printed_page_vlm", "vlm"), ("printed_page_textlayer", "textlayer")):
            value = str(row.get(key) or "").strip()
            if value.isdigit() and abs(int(value) - expected) <= 2:
                chosen, source = int(value), label
                break
        row["printed_page"] = chosen if chosen is not None else expected
        row["printed_page_source"] = source
    return modal_offset


## 6. Figure detection (geometric, no model)

Two detectors, because a scanned PDF can store figures either way and it is not knowable
without looking at this file:

- **`embedded`** — real image XObjects, via `get_image_rects`. Rects covering more than
  `FIG_MAX_AREA_FRAC` of the page are dropped: on an MRC-compressed scan those are the
  background/foreground layers of the page itself, not figures.
- **`textgap`** — vertical bands containing no text blocks. Works when the figures are
  baked into one flat page image, but misses a figure that sits *beside* wrapped text.

`auto` tries `embedded` and falls back to `textgap`. Every candidate is then tightened
to its ink bounding box, which trims whitespace and rejects blank bands.

**Run section 6.1 before committing to a full run** — it shows you which detector fires
on your file and what it actually crops.


In [ ]:
def _rect(obj):
    return obj if isinstance(obj, fitz.Rect) else fitz.Rect(obj)


def merge_boxes(boxes, pad=3.0):
    boxes = [_rect(b) for b in boxes]
    changed = True
    while changed and boxes:
        changed = False
        merged = []
        for box in boxes:
            placed = False
            for i, other in enumerate(merged):
                if other.intersects(box + (-pad, -pad, pad, pad)):
                    merged[i] = other | box
                    placed = True
                    changed = True
                    break
            if not placed:
                merged.append(box)
        boxes = merged
    return boxes


def figure_boxes_embedded(page):
    page_area = abs(page.rect.get_area()) or 1.0
    boxes = []
    for info in page.get_images(full=True):
        try:
            rects = page.get_image_rects(info[0]) or []
        except Exception:
            continue
        for rect in rects:
            rect = _rect(rect)
            if rect.is_empty or rect.is_infinite:
                continue
            if abs(rect.get_area()) / page_area > FIG_MAX_AREA_FRAC:
                continue  # page scan layer, not a figure
            if rect.width / page.rect.width < FIG_MIN_WIDTH_FRAC:
                continue
            if rect.height / page.rect.height < FIG_MIN_HEIGHT_FRAC:
                continue
            boxes.append(rect)
    return merge_boxes(boxes)


def figure_boxes_textgap(page):
    blocks = [_rect(b[:4]) for b in page.get_text("blocks") if b[6] == 0 and str(b[4]).strip()]
    if len(blocks) < 2:
        return []
    left = min(b.x0 for b in blocks)
    right = max(b.x1 for b in blocks)
    top = min(b.y0 for b in blocks)
    bottom = max(b.y1 for b in blocks)

    occupied = []
    for block in sorted(blocks, key=lambda b: b.y0):
        if occupied and block.y0 <= occupied[-1][1] + TEXT_LINE_MERGE_PTS:
            occupied[-1][1] = max(occupied[-1][1], block.y1)
        else:
            occupied.append([block.y0, block.y1])

    gaps = []
    cursor = top
    for y0, y1 in occupied:
        if y0 - cursor >= FIG_MIN_GAP_PTS:
            gaps.append((cursor, y0))
        cursor = max(cursor, y1)
    if bottom - cursor >= FIG_MIN_GAP_PTS:
        gaps.append((cursor, bottom))

    return [fitz.Rect(left, y0, right, y1) for y0, y1 in gaps]


def tighten_to_ink(page, rect, probe_dpi=110):
    """Shrink a candidate box to its ink. Returns (rect, ink_fraction) or (None, frac)."""
    rect = _rect(rect) & page.rect
    if rect.is_empty or rect.width < 4 or rect.height < 4:
        return None, 0.0
    zoom = probe_dpi / 72.0
    pix = page.get_pixmap(clip=rect, matrix=fitz.Matrix(zoom, zoom),
                          colorspace=fitz.csGRAY, alpha=False)
    if pix.height < 2 or pix.width < 2:
        return None, 0.0
    arr = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)[:, :, 0]
    ink = arr < INK_THRESHOLD
    ink_fraction = float(ink.mean())
    if ink_fraction < FIG_MIN_INK_FRAC:
        return None, ink_fraction

    rows = np.where(ink.mean(axis=1) > 0.004)[0]
    cols = np.where(ink.mean(axis=0) > 0.004)[0]
    if rows.size == 0 or cols.size == 0:
        return None, ink_fraction

    scale = 72.0 / probe_dpi
    tight = fitz.Rect(
        rect.x0 + cols[0] * scale, rect.y0 + rows[0] * scale,
        rect.x0 + (cols[-1] + 1) * scale, rect.y0 + (rows[-1] + 1) * scale,
    )
    tight = (tight + (-FIG_PAD_PTS, -FIG_PAD_PTS, FIG_PAD_PTS, FIG_PAD_PTS)) & page.rect
    if tight.width / page.rect.width < FIG_MIN_WIDTH_FRAC:
        return None, ink_fraction
    if tight.height / page.rect.height < FIG_MIN_HEIGHT_FRAC:
        return None, ink_fraction
    return tight, ink_fraction


FIG_LABEL_RE = re.compile(r"\bFig(?:ure)?\.?\s*([0-9]{1,4}[a-zA-Z]?)", re.IGNORECASE)


def _text_in_zone(page, zone):
    parts = []
    for block in page.get_text("blocks"):
        if block[6] != 0:
            continue
        text = str(block[4]).strip()
        if text and _rect(block[:4]).intersects(zone):
            parts.append(text)
    return clean_string(" ".join(parts))


def figure_context(page, rect):
    """The printed label above the cut and the printed legend below it.

    This book prints 'Fig. 1.' above the engraving and the lettered key beneath it.
    Both are author-written text, so they are captured verbatim, never generated.
    """
    above = fitz.Rect(rect.x0 - 24, max(page.rect.y0, rect.y0 - FIG_LABEL_LOOKBACK_PTS),
                      rect.x1 + 24, rect.y0) & page.rect
    below = fitz.Rect(rect.x0 - 24, rect.y1,
                      rect.x1 + 24, min(page.rect.y1, rect.y1 + FIG_LEGEND_LOOKAHEAD_PTS)) & page.rect

    label_text = _text_in_zone(page, above)
    legend_text = _text_in_zone(page, below)
    match = FIG_LABEL_RE.search(label_text) or FIG_LABEL_RE.search(legend_text)
    label = f"Fig. {match.group(1)}" if match else ""

    legend_rect = None
    if legend_text:
        for block in page.get_text("blocks"):
            if block[6] == 0 and str(block[4]).strip() and _rect(block[:4]).intersects(below):
                legend_rect = _rect(block[:4]) if legend_rect is None else (legend_rect | _rect(block[:4]))
    return {
        "figure_label": label,
        "printed_legend": legend_text,
        "label_text_above": label_text,
        "legend_rect": legend_rect,
    }


def detect_figures(pdf_file, page_number_in_file, book_page):
    """Geometric only. Returns a list of dicts with crop paths and printed text."""
    page = get_doc(pdf_file).load_page(page_number_in_file - 1)
    mode = FIGURE_DETECTION_MODE
    if mode in ("auto", "embedded"):
        candidates = figure_boxes_embedded(page)
        used = "embedded"
        if not candidates and mode == "auto":
            candidates = figure_boxes_textgap(page)
            used = "textgap"
    else:
        candidates = figure_boxes_textgap(page)
        used = "textgap"

    out_dir = FIGURE_DIR / safe_slug(pdf_file)
    out_dir.mkdir(parents=True, exist_ok=True)
    zoom = RENDER_DPI / 72.0
    figures = []

    for rect in sorted(candidates, key=lambda r: (r.y0, r.x0)):
        tight, ink_fraction = tighten_to_ink(page, rect)
        if tight is None:
            continue
        context = figure_context(page, tight)

        # The crop sent to the model includes the printed legend, so the model can
        # read it; the crop kept for citation is the figure alone.
        with_legend = tight
        if context["legend_rect"] is not None:
            with_legend = (tight | context["legend_rect"]) & page.rect

        index = len(figures) + 1
        stem = f"p{book_page:04d}_fig{index:02d}"
        crop_path = out_dir / f"{stem}.png"
        context_path = out_dir / f"{stem}_ctx.png"

        for target_rect, target_path in ((tight, crop_path), (with_legend, context_path)):
            pix = page.get_pixmap(clip=target_rect, matrix=fitz.Matrix(zoom, zoom), alpha=False)
            img = Image.open(BytesIO(pix.tobytes("png"))).convert("RGB")
            resize_max_side(img, FIGURE_MAX_SIDE).save(target_path, optimize=True)

        figures.append({
            "figure_id": f"{BOOK_SLUG}_{stem}",
            "source_pdf": Path(pdf_file).name,
            "pdf_page": book_page,
            "figure_index": index,
            "figure_label": context["figure_label"],
            "printed_legend": context["printed_legend"],
            "detector": used,
            "bbox_pts": [round(v, 2) for v in (tight.x0, tight.y0, tight.x1, tight.y1)],
            "page_area_fraction": round(abs(tight.get_area()) / max(1.0, abs(page.rect.get_area())), 4),
            "ink_fraction": round(ink_fraction, 4),
            "crop_path": str(crop_path),
            "crop_with_legend_path": str(context_path),
        })
        if len(figures) >= MAX_FIGURES_PER_PAGE:
            break
    return figures


### 6.1 Detector diagnostic — run this before the full run

Prints which detector fired, the printed label and legend it harvested, and shows each
crop. If the crops are wrong, tune `FIG_*` in section 1 rather than discovering it 400
pages into a GPU session.


In [ ]:
DIAGNOSTIC_PAGES = [13, 16, 19, 20, 21]   # book pages; the samples you already reviewed

for book_page in DIAGNOSTIC_PAGES:
    target = None
    for pdf in PDF_FILES:
        offset = pdf_page_offset(pdf)
        local = book_page - offset
        if 1 <= local <= get_doc(pdf).page_count:
            target = (pdf, local)
            break
    if target is None:
        print(f"book page {book_page}: not in the loaded PDFs")
        continue

    pdf_file, local_page = target
    page = get_doc(pdf_file).load_page(local_page - 1)
    furniture = page_furniture_from_textlayer(page)
    figures = detect_figures(pdf_file, local_page, book_page)

    print("=" * 78)
    print(f"book page {book_page} ({pdf_file.name} p{local_page}) | "
          f"printed_page_textlayer={furniture['printed_page_textlayer']!r} | "
          f"running_head={furniture['running_head_textlayer']!r}")
    print(f"  embedded candidates: {len(figure_boxes_embedded(page))} | "
          f"textgap candidates: {len(figure_boxes_textgap(page))} | kept: {len(figures)}")
    for figure in figures:
        print(f"  - {figure['figure_id']} [{figure['detector']}] "
              f"label={figure['figure_label']!r} area={figure['page_area_fraction']} "
              f"ink={figure['ink_fraction']}")
        print(f"    legend: {figure['printed_legend'][:180]!r}")
        display(resize_max_side(Image.open(figure["crop_path"]), 420))


## 7. Prompts

Two prompts, both narrow.

The page prompt transcribes; it never describes an illustration and never decides a page
is worthless. Faithfulness is stated explicitly because the failure mode of a VLM on an
1875 book is *silent modernisation* — quietly turning `diarrhœa` into `diarrhea` and
`Spermatozoön` into `Spermatozoon`. That looks clean and would score well against itself,
which is exactly why it must be forbidden here.

The figure prompt sees one crop plus its printed legend and the paragraph that references
it. `legend_agreement` is the cheap hallucination check: the book already told us what the
picture is, so a description that contradicts the legend is detectable without a human.


In [ ]:
OCR_SCHEMA = r"""
Return only one valid JSON object:
{
  "page_type": "cover|front_matter|contents|index|advertisement|blank|prose|figure_plate|table|mixed",
  "printed_page_number": "",
  "running_head": "",
  "chapter": "",
  "section": "",
  "headings": [""],
  "clean_text": "faithful transcription in reading order",
  "figure_legends": [{"label": "Fig. 1", "legend": "caption exactly as printed"}],
  "tables_markdown": [""],
  "is_blank": false,
  "quality_flags": ["foxing|faded_ink|bleed_through|show_through|skew|blur|cropped_text|uncertain_text"],
  "uncertain_spans": [""]
}
""".strip()


def make_ocr_prompt(book_page, source_pdf):
    return f"""You are a high-accuracy OCR system for 19th-century printed English medical books.
Transcribe page {book_page} of "{BOOK_TITLE}" (file: {source_pdf}) exactly as it is printed.

TRANSCRIBE EVERY PAGE. Covers, prefaces, contents, indexes, advertisements and near-blank
pages all get transcribed. Record what kind of page it is in page_type. Never return an
empty clean_text because a page seems unimportant. Only a page with no printed characters
at all may have empty clean_text, and then set is_blank=true.

Rules:
1. Natural reading order. The book is normally single-column; if a page is genuinely set
   in two columns, transcribe the left column fully, then the right.
2. Reproduce the text EXACTLY as printed. Do not modernise spelling, hyphenation,
   capitalisation, italics-as-words, or punctuation. Keep period forms and diacritics as
   printed: "diarrhoea" stays "diarrhoea" only if that is what is printed; if the page
   prints "diarrhoea" with an oe-ligature or "Spermatozoon" with a diaeresis, keep the
   printed character. Do not silently correct the author.
3. Preserve line breaks within a paragraph, and put a blank line between paragraphs. Keep
   a word broken across a line with its hyphen, as printed. Do not join it.
4. Do not translate, summarise, paraphrase, explain, answer, or add anything not printed.
   Do not correct factual or medical errors; this is a historical document.
5. Never guess an unreadable character or word. Write [illegible] and list the surrounding
   phrase in uncertain_spans.
6. The running head, the printed page number, signature marks and catchwords are page
   furniture. Put them in their own fields, not in clean_text.
7. Do NOT describe illustrations. For each illustration, copy its printed caption or key
   verbatim into figure_legends, and write a marker line [FIGURE: Fig. N] on its own line
   at the point in clean_text where the illustration sits. Use [FIGURE: unlabelled] if the
   illustration has no printed figure number.
8. Convert a printed table to Markdown in tables_markdown, and put a [TABLE 1] marker line
   at its position in clean_text.
9. Report honest quality_flags. Brown staining is foxing; text from the reverse side
   showing through is show_through.

{OCR_SCHEMA}"""


FIGURE_SCHEMA = r"""
Return only one valid JSON object:
{
  "figure_type": "anatomical_diagram|botanical_illustration|portrait|apparatus|chart_or_table|map|decorative|photograph|unclear",
  "legend_ocr": "the caption text visible in this crop, verbatim, or empty",
  "visible_labels": ["A", "B", "1", "2"],
  "description": "60-150 words describing only what is drawn",
  "depicted_entities": [""],
  "legend_agreement": "agrees|partial|contradicts|no_legend",
  "confidence": "high|medium|low",
  "quality_flags": ["faded|low_contrast|cropped|overlapping_text|too_small_to_read"]
}
""".strip()


def make_figure_prompt(figure, page_context_text):
    printed_legend = figure.get("printed_legend", "") or "[no legend found in the page text layer]"
    label = figure.get("figure_label", "") or "unlabelled"
    context = (page_context_text or "").strip()[:900] or "[no page text available]"
    return f"""You are describing ONE illustration cropped from page {figure.get('pdf_page')} of
"{BOOK_TITLE}", an American medical book printed in the 1870s-1890s. The crop may include
the caption printed beneath the illustration.

Figure label from the page: {label}
Caption as recorded in the page text layer (may itself contain OCR errors):
{printed_legend}

Surrounding page text, for orientation only:
{context}

Rules:
1. Describe ONLY what is visibly drawn in this crop. Do not add anatomy, botany, medicine
   or history that you know but cannot see. Do not diagnose or interpret clinically.
2. Do not repeat the caption as your description. The caption is already stored separately.
   Your description must add what the caption does not say: layout, orientation, what each
   labelled part looks like, shading, cross-section vs whole view, how many objects.
3. visible_labels must contain only letters or numbers you can actually see printed on the
   illustration. If you cannot read any, return an empty list. Never invent a label.
4. Set legend_agreement by comparing what you see against the caption above:
   "agrees" if the drawing matches the caption, "partial" if the caption covers only some
   of what is drawn, "contradicts" if the drawing clearly is not what the caption says,
   "no_legend" if there is no usable caption.
5. If the crop is mostly blank, a page ornament, or a rule/border rather than a real
   illustration, say so plainly and set figure_type to "decorative" or "unclear".
6. This is a historical source. Describe its imagery neutrally and factually.

{FIGURE_SCHEMA}"""


def image_to_data_url(image):
    buffer = BytesIO()
    image.save(buffer, format="PNG", optimize=True)
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/png;base64,{encoded}"


def extract_json_object(text):
    raw = str(text or "").strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    try:
        return json.loads(raw)
    except Exception:
        start, end = raw.find("{"), raw.rfind("}")
        if start >= 0 and end > start:
            return json.loads(raw[start:end + 1])
    raise ValueError("No valid JSON object found in model output")


def as_string_list(value):
    if not isinstance(value, list):
        return []
    return [clean_string(x) for x in value if clean_string(x)]


## 8. Load Qwen3.5 GGUF and the multimodal projector


In [ ]:
try:
    from llama_cpp.llama_chat_format import MTMDChatHandler
except ImportError as exc:
    raise ImportError(
        "This llama-cpp-python build lacks MTMDChatHandler. Use a recent multimodal wheel."
    ) from exc

print("Loading model and multimodal projector...")
chat_handler = MTMDChatHandler(clip_model_path=str(MMPROJ_FILE), verbose=False)
llm = Llama(
    model_path=str(MODEL_FILE),
    chat_handler=chat_handler,
    n_ctx=N_CTX,
    n_gpu_layers=N_GPU_LAYERS,
    n_batch=N_BATCH,
    n_threads=N_THREADS,
    verbose=False,
)
print("Model loaded.")


## 9. Model calls


In [ ]:
@contextlib.contextmanager
def suppress_native_output(enabled=True):
    if not enabled:
        yield
        return
    sys.stdout.flush()
    sys.stderr.flush()
    saved_stdout_fd = os.dup(1)
    saved_stderr_fd = os.dup(2)
    try:
        with open(os.devnull, "w") as devnull:
            os.dup2(devnull.fileno(), 1)
            os.dup2(devnull.fileno(), 2)
            with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
                yield
    finally:
        sys.stdout.flush()
        sys.stderr.flush()
        os.dup2(saved_stdout_fd, 1)
        os.dup2(saved_stderr_fd, 2)
        os.close(saved_stdout_fd)
        os.close(saved_stderr_fd)


def response_usage(response, max_tokens):
    usage = response.get("usage", {}) or {}
    completion = int(usage.get("completion_tokens", 0) or 0)
    return {
        "prompt_tokens": int(usage.get("prompt_tokens", 0) or 0),
        "completion_tokens": completion,
        "total_tokens": int(usage.get("total_tokens", 0) or 0),
        "hit_token_limit": completion >= max(1, max_tokens - 8),
    }


def run_chat(messages, max_tokens, quiet=True):
    started = time.perf_counter()
    with suppress_native_output(quiet):
        response = llm.create_chat_completion(
            messages=messages,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=max_tokens,
        )
    elapsed = time.perf_counter() - started
    raw = response["choices"][0]["message"]["content"].strip()
    return raw, {**response_usage(response, max_tokens), "elapsed_seconds": round(elapsed, 3)}


def vision_ocr_page(image, book_page, source_pdf, furniture):
    raw, usage = run_chat(
        [{
            "role": "user",
            "content": [
                {"type": "text", "text": make_ocr_prompt(book_page, source_pdf)},
                {"type": "image_url", "image_url": {"url": image_to_data_url(image)}},
            ],
        }],
        MAX_OCR_TOKENS,
    )
    data = extract_json_object(raw)

    clean_text = str(data.get("clean_text", "")).strip()
    legends = []
    for item in data.get("figure_legends", []) if isinstance(data.get("figure_legends"), list) else []:
        if isinstance(item, dict):
            legends.append({"label": clean_string(item.get("label")), "legend": clean_string(item.get("legend"))})
        elif clean_string(item):
            legends.append({"label": "", "legend": clean_string(item)})

    return {
        "source_pdf": source_pdf,
        "page": int(book_page),                 # absolute book page; resume key
        "pdf_page": int(book_page),
        "page_type": clean_string(data.get("page_type", "prose")).lower() or "prose",
        "is_blank": bool(data.get("is_blank", False)),
        "printed_page_vlm": clean_string(data.get("printed_page_number")),
        "printed_page_textlayer": furniture.get("printed_page_textlayer", ""),
        "running_head": clean_string(data.get("running_head")) or furniture.get("running_head_textlayer", ""),
        "running_head_textlayer": furniture.get("running_head_textlayer", ""),
        "chapter": clean_string(data.get("chapter")),
        "section": clean_string(data.get("section")),
        "headings": as_string_list(data.get("headings")),
        "clean_text": clean_text,                              # faithful; score OCR on this
        "normalized_text": normalize_historical_text(clean_text),  # embed this
        "figure_legends": legends,
        "tables_markdown": as_string_list(data.get("tables_markdown")),
        "quality_flags": as_string_list(data.get("quality_flags")),
        "uncertain_spans": as_string_list(data.get("uncertain_spans")),
        "char_count": len(clean_text),
        "word_count": len(clean_text.split()),
        "raw_model_output": raw,
        "usage": usage,
    }


def describe_figure(figure, page_context_text):
    image = Image.open(figure["crop_with_legend_path"]).convert("RGB")
    raw, usage = run_chat(
        [{
            "role": "user",
            "content": [
                {"type": "text", "text": make_figure_prompt(figure, page_context_text)},
                {"type": "image_url", "image_url": {"url": image_to_data_url(image)}},
            ],
        }],
        MAX_FIGURE_TOKENS,
    )
    data = extract_json_object(raw)
    description = str(data.get("description", "")).strip()
    record = dict(figure)
    record.update({
        "figure_type": clean_string(data.get("figure_type", "unclear")),
        "legend_ocr_vlm": clean_string(data.get("legend_ocr")),
        "visible_labels": as_string_list(data.get("visible_labels")),
        "description": description,
        "depicted_entities": as_string_list(data.get("depicted_entities")),
        "legend_agreement": clean_string(data.get("legend_agreement", "no_legend")),
        "confidence": clean_string(data.get("confidence", "medium")),
        "figure_quality_flags": as_string_list(data.get("quality_flags")),
        "raw_model_output": raw,
        "usage": usage,
    })
    return record


def figure_retrieval_text(record):
    """Built in Python, not generated: the model is never asked to write the citation."""
    parts = []
    header = " ".join(x for x in [record.get("figure_label", ""), record.get("section", "")] if x)
    if header:
        parts.append(header)
    if record.get("printed_legend"):
        parts.append("Printed caption: " + record["printed_legend"])
    if record.get("description"):
        parts.append("Depicted: " + record["description"])
    labels = record.get("visible_labels") or []
    if labels:
        parts.append("Labelled parts: " + ", ".join(labels))
    parts.append(f"Source: {BOOK_TITLE}, printed page {record.get('printed_page', record.get('pdf_page'))}.")
    return "\n".join(parts)


def vision_ocr_crop(crop_image, book_page, source_pdf, block_label="text"):
    prompt = f"""Transcribe this text region ({block_label}) from page {book_page} of "{BOOK_TITLE}".
Reproduce the printed characters EXACTLY. Keep original spelling and line breaks. Return only the raw transcribed text."""
    raw, usage = run_chat(
        [{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": image_to_data_url(crop_image)}},
            ],
        }],
        MAX_OCR_TOKENS,
        quiet=True,
    )
    raw_clean = re.sub(r"^```(?:json|text)?\s*", "", raw)
    raw_clean = re.sub(r"\s*```$", "", raw_clean).strip()
    return raw_clean, usage


def vision_ocr_page_layout_aware(image, book_page, source_pdf, furniture, chandra_blocks_map=None):
    page_id = f"p{int(book_page):04d}"
    blocks = chandra_blocks_map.get(page_id, []) if chandra_blocks_map else []
    
    if USE_CHANDRA_LAYOUT and blocks:
        img_w, img_h = image.size
        img_np = np.array(image)
        block_texts = []
        total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0, "elapsed_seconds": 0.0}
        
        for blk in blocks:
            px_res = bbox_to_pixel(blk.get("bbox"), blk.get("page_box"), img_w, img_h)
            if px_res is None:
                continue
            px0, py0, px1, py1 = px_res
            if px1 <= px0 or py1 <= py0:
                continue
            crop_np = img_np[py0:py1, px0:px1]
            if crop_np.size == 0:
                continue
            crop_img = Image.fromarray(crop_np)
            text, u = vision_ocr_crop(crop_img, book_page, source_pdf, block_label=blk.get("label", "text"))
            if text:
                block_texts.append(text)
            for k in ["prompt_tokens", "completion_tokens", "total_tokens"]:
                total_usage[k] += u.get(k, 0)
            total_usage["elapsed_seconds"] += u.get("elapsed_seconds", 0.0)
            
        clean_text = "\n".join(block_texts)
        return {
            "source_pdf": source_pdf,
            "page": int(book_page),
            "pdf_page": int(book_page),
            "page_type": "prose",
            "is_blank": not bool(clean_text),
            "printed_page_vlm": "",
            "printed_page_textlayer": furniture.get("printed_page_textlayer", ""),
            "running_head": furniture.get("running_head_textlayer", ""),
            "running_head_textlayer": furniture.get("running_head_textlayer", ""),
            "chapter": "",
            "section": "",
            "headings": [],
            "clean_text": clean_text,
            "normalized_text": normalize_historical_text(clean_text),
            "figure_legends": [],
            "tables_markdown": [],
            "quality_flags": [],
            "uncertain_spans": [],
            "char_count": len(clean_text),
            "word_count": len(clean_text.split()),
            "raw_model_output": clean_text,
            "usage": total_usage,
            "layout_blocks_count": len(block_texts),
        }
    else:
        # Fallback to full-page OCR
        return vision_ocr_page(image, book_page, source_pdf, furniture)


## 10. Checkpoints, deterministic chunking, exports

Chunking replaces the second model call. It is paragraph-aware, carries a paragraph that
runs across a page break (so a sentence split by the printer is not split again by the
index), resets at a section change, and records `pdf_page_start`/`pdf_page_end` so every
chunk can be cited to a real printed page.


In [ ]:
def append_jsonl(path, item):
    with Path(path).open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(item, ensure_ascii=False) + "\n")


def load_latest(path, key_fields=("source_pdf", "page")):
    """Last line wins, so a retried page never duplicates a final record."""
    latest = {}
    path = Path(path)
    if not path.exists():
        return latest
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except Exception:
                continue
            latest[tuple(str(item.get(f, "")) for f in key_fields)] = item
    return latest


def page_key(source_pdf, book_page):
    return (str(source_pdf), str(int(book_page)))


PAGE_QUALITY = {}
if PAGE_QUALITY_CSV and Path(PAGE_QUALITY_CSV).exists():
    _q = pd.read_csv(PAGE_QUALITY_CSV)
    PAGE_QUALITY = {page_key(r["source_pdf"], r["page"]): float(r["ocr_quality"]) for _, r in _q.iterrows()}
    print(f"Loaded {len(PAGE_QUALITY)} page quality scores; routing pages at or below {PAGE_QUALITY_THRESHOLD}.")


def needs_vlm(source_pdf, book_page):
    if not PAGE_QUALITY:
        return True
    return PAGE_QUALITY.get(page_key(source_pdf, book_page), 0.0) <= PAGE_QUALITY_THRESHOLD


SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z\"'(])")
MARKER_RE = re.compile(r"^\s*\[(FIGURE|TABLE)[^\]]*\]\s*$", re.IGNORECASE)


def split_paragraphs(text):
    paragraphs = []
    for block in re.split(r"\n\s*\n", str(text or "")):
        block = re.sub(r"[ \t]+", " ", block).strip()
        if block and not MARKER_RE.match(block):
            paragraphs.append(block)
    return paragraphs


def _split_long_paragraph(paragraph):
    sentences = SENTENCE_SPLIT_RE.split(paragraph)
    pieces, current = [], ""
    for sentence in sentences:
        if current and len(current) + len(sentence) + 1 > CHUNK_MAX_CHARS:
            pieces.append(current.strip())
            current = sentence
        else:
            current = f"{current} {sentence}".strip()
    if current.strip():
        pieces.append(current.strip())
    return pieces


def build_chunks(page_rows):
    """page_rows: successful page dicts, sorted by (source_pdf, pdf_page)."""
    chunks = []
    buffer = []          # list of (paragraph, page_start, page_end)
    buffer_meta = None
    carry_tail = ""      # trailing text of the previous chunk, for overlap

    def flush():
        nonlocal buffer
        if not buffer or buffer_meta is None:
            buffer = []
            return
        current, starts, ends = [], [], []
        for paragraph, page_start, page_end in buffer:
            for piece in ([paragraph] if len(paragraph) <= CHUNK_MAX_CHARS else _split_long_paragraph(paragraph)):
                joined_len = sum(len(p) for p in current) + len(piece)
                if current and joined_len > CHUNK_TARGET_CHARS:
                    emit(current, starts, ends)
                    current, starts, ends = [], [], []
                current.append(piece)
                starts.append(page_start)
                ends.append(page_end)
        if current:
            emit(current, starts, ends)
        buffer = []

    def emit(pieces, starts, ends):
        nonlocal carry_tail
        body = "\n\n".join(pieces).strip()
        # A paragraph merged across a page break carries both its pages, so a chunk
        # containing it cites the real range and not just the page it started on.
        page_start, page_end = min(starts), max(ends)
        printed = buffer_meta["printed_page_by_pdf"]
        if len(body) < CHUNK_MIN_CHARS and chunks:
            # Too small to stand alone: append to the previous chunk instead.
            previous = chunks[-1]
            previous["text"] = (previous["text"] + "\n\n" + body).strip()
            previous["pdf_page_end"] = max(previous["pdf_page_end"], page_end)
            previous["printed_page_end"] = printed.get(previous["pdf_page_end"], previous["pdf_page_end"])
            previous["char_count"] = len(previous["text"])
            previous["word_count"] = len(previous["text"].split())
            return
        overlap = carry_tail[-CHUNK_OVERLAP_CHARS:] if CHUNK_OVERLAP_CHARS and carry_tail else ""
        text = (overlap + "\n\n" + body).strip() if overlap else body
        index = len(chunks) + 1
        chunks.append({
            "chunk_id": f"{BOOK_SLUG}_p{page_start:04d}_c{index:04d}",
            "modality": "text",
            "source_pdf": buffer_meta["source_pdf"],
            "pdf_page_start": page_start,
            "pdf_page_end": page_end,
            "printed_page_start": printed.get(page_start, page_start),
            "printed_page_end": printed.get(page_end, page_end),
            "page_type": buffer_meta["page_type"],
            "running_head": buffer_meta["running_head"],
            "chapter": buffer_meta["chapter"],
            "section": buffer_meta["section"],
            "text": text,
            "char_count": len(text),
            "word_count": len(text.split()),
            "overlap_chars": len(overlap),
        })
        carry_tail = body

    printed_by_pdf = {}
    previous_section = None
    previous_source = None

    for row in page_rows:
        printed_by_pdf[int(row["pdf_page"])] = row.get("printed_page", row["pdf_page"])
        section_key = (row.get("source_pdf"), row.get("running_head", ""), row.get("chapter", ""))
        if row.get("page_type") not in CHUNKABLE_PAGE_TYPES:
            # An index or advertisement page ends the run of prose. Resetting the
            # tail here too, otherwise the overlap bleeds across it into the next
            # chapter and chunks quote text from a section they do not belong to.
            flush()
            carry_tail = ""
            previous_section = None
            continue
        if previous_section is not None and section_key != previous_section:
            flush()
            carry_tail = ""
        if previous_source is not None and row.get("source_pdf") != previous_source:
            flush()
            carry_tail = ""

        buffer_meta = {
            "source_pdf": row.get("source_pdf", ""),
            "page_type": row.get("page_type", ""),
            "running_head": row.get("running_head", ""),
            "chapter": row.get("chapter", ""),
            "section": row.get("section", "") or row.get("running_head", ""),
            "printed_page_by_pdf": printed_by_pdf,
        }

        paragraphs = split_paragraphs(row.get("normalized_text") or row.get("clean_text"))
        page_no = int(row["pdf_page"])
        if paragraphs and buffer:
            last_paragraph, last_start, _last_end = buffer[-1]
            starts_lower = paragraphs[0][:1].islower()
            unfinished = not last_paragraph.rstrip().endswith(TERMINAL_PUNCT)
            if starts_lower and unfinished:
                # A paragraph the printer broke across the page break. Keep both
                # page numbers so the merged text stays citable to its real range.
                buffer[-1] = (last_paragraph.rstrip() + " " + paragraphs.pop(0), last_start, page_no)
        buffer.extend((p, page_no, page_no) for p in paragraphs)

        previous_section = section_key
        previous_source = row.get("source_pdf")
        if sum(len(entry[0]) for entry in buffer) >= CHUNK_MAX_CHARS * 3:
            flush()

    flush()
    return chunks


def save_exports(verbose=False):
    ocr_map = load_latest(PAGE_OCR_JSONL)
    figure_map = load_latest(FIGURES_JSONL, key_fields=("figure_id",))

    page_rows = [item for item in ocr_map.values() if not item.get("error")]
    page_rows.sort(key=lambda r: (str(r.get("source_pdf", "")), int(r.get("pdf_page", 0))))
    modal_offset = reconcile_printed_pages(page_rows) if page_rows else 0
    printed_by_page = {(r["source_pdf"], int(r["pdf_page"])): r["printed_page"] for r in page_rows}
    section_by_page = {(r["source_pdf"], int(r["pdf_page"])): (r.get("section") or r.get("running_head", ""))
                       for r in page_rows}

    pages_df = pd.DataFrame([{
        "source_pdf": r.get("source_pdf"), "pdf_page": r.get("pdf_page"),
        "printed_page": r.get("printed_page"), "printed_page_source": r.get("printed_page_source"),
        "page_type": r.get("page_type"), "is_blank": r.get("is_blank"),
        "running_head": r.get("running_head"), "chapter": r.get("chapter"), "section": r.get("section"),
        "char_count": r.get("char_count"), "word_count": r.get("word_count"),
        "n_figure_legends": len(r.get("figure_legends", []) or []),
        "quality_flags": "|".join(r.get("quality_flags", []) or []),
        "n_uncertain_spans": len(r.get("uncertain_spans", []) or []),
    } for r in page_rows])
    if not pages_df.empty:
        pages_df.to_csv(PAGES_CSV, index=False, encoding="utf-8-sig")

    figure_rows = []
    for record in figure_map.values():
        if record.get("error"):
            continue
        record = dict(record)
        key = (record.get("source_pdf"), int(record.get("pdf_page", 0)))
        record["printed_page"] = printed_by_page.get(key, record.get("pdf_page"))
        record["section"] = section_by_page.get(key, "")
        record["retrieval_text"] = figure_retrieval_text(record)
        figure_rows.append(record)
    figure_rows.sort(key=lambda r: (str(r.get("source_pdf", "")), int(r.get("pdf_page", 0)), int(r.get("figure_index", 0))))

    if figure_rows:
        pd.DataFrame([{
            "figure_id": f["figure_id"], "source_pdf": f["source_pdf"],
            "pdf_page": f["pdf_page"], "printed_page": f["printed_page"],
            "figure_label": f.get("figure_label", ""), "figure_type": f.get("figure_type", ""),
            "printed_legend": f.get("printed_legend", ""), "description": f.get("description", ""),
            "legend_agreement": f.get("legend_agreement", ""), "confidence": f.get("confidence", ""),
            "detector": f.get("detector", ""), "crop_path": f.get("crop_path", ""),
        } for f in figure_rows]).to_csv(FIGURES_CSV, index=False, encoding="utf-8-sig")

    chunks = build_chunks(page_rows)
    with CHUNKS_JSONL.open("w", encoding="utf-8") as handle:
        for chunk in chunks:
            handle.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    corpus = []
    for chunk in chunks:
        row = dict(chunk)
        row["retrieval_text"] = chunk["text"]
        row["citation"] = f"{BOOK_TITLE}, p. {chunk['printed_page_start']}"
        corpus.append(row)
    for figure in figure_rows:
        corpus.append({
            "chunk_id": figure["figure_id"],
            "modality": "figure",
            "source_pdf": figure["source_pdf"],
            "pdf_page_start": figure["pdf_page"], "pdf_page_end": figure["pdf_page"],
            "printed_page_start": figure["printed_page"], "printed_page_end": figure["printed_page"],
            "page_type": "figure", "running_head": "", "chapter": "",
            "section": figure.get("section", ""),
            "text": figure.get("description", ""),
            "retrieval_text": figure["retrieval_text"],
            "figure_label": figure.get("figure_label", ""),
            "figure_type": figure.get("figure_type", ""),
            "printed_legend": figure.get("printed_legend", ""),
            "legend_agreement": figure.get("legend_agreement", ""),
            "crop_path": figure.get("crop_path", ""),
            "char_count": len(figure.get("description", "")),
            "word_count": len(figure.get("description", "").split()),
            "citation": f"{BOOK_TITLE}, p. {figure['printed_page']} ({figure.get('figure_label') or 'figure'})",
        })

    with RAG_CORPUS_JSONL.open("w", encoding="utf-8") as handle:
        for row in corpus:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    corpus_df = pd.DataFrame(corpus)
    corpus_df.to_csv(RAG_CORPUS_CSV, index=False, encoding="utf-8-sig")

    token_rows, error_rows = [], []
    for stage, mapping in (("page_ocr", ocr_map), ("figure", figure_map)):
        for item in mapping.values():
            usage = item.get("usage", {}) or {}
            token_rows.append({
                "stage": stage, "source_pdf": item.get("source_pdf", ""),
                "page": item.get("pdf_page", item.get("page", "")),
                "figure_id": item.get("figure_id", ""), **usage,
            })
            if item.get("error"):
                error_rows.append({
                    "stage": stage, "source_pdf": item.get("source_pdf", ""),
                    "page": item.get("pdf_page", item.get("page", "")),
                    "figure_id": item.get("figure_id", ""), "error": item["error"],
                })
    pd.DataFrame(token_rows).to_csv(TOKEN_LOG_CSV, index=False, encoding="utf-8-sig")
    pd.DataFrame(error_rows, columns=["stage", "source_pdf", "page", "figure_id", "error"]).to_csv(
        ERROR_LOG_CSV, index=False, encoding="utf-8-sig")

    with READABLE_TXT.open("w", encoding="utf-8") as handle:
        for row in corpus:
            handle.write(f"[{row['chunk_id']}] ({row['modality']}) {row.get('citation', '')}\n")
            handle.write(str(row.get("retrieval_text", "")).strip() + "\n\n")

    if verbose:
        print(f"  printed-page offset (printed - pdf) = {modal_offset}")
    return pages_df, pd.DataFrame(figure_rows), corpus_df


if RESET_OUTPUTS:
    for path in [PAGE_OCR_JSONL, FIGURES_JSONL, CHUNKS_JSONL, RAG_CORPUS_JSONL, RAG_CORPUS_CSV,
                 PAGES_CSV, FIGURES_CSV, TOKEN_LOG_CSV, ERROR_LOG_CSV, READABLE_TXT]:
        if path.exists():
            path.unlink()
            print("Removed:", path)


## 11. Run

Per page: one vision call for the text, then one small vision call per detected figure.
Figure detection itself is geometric and runs even if `DESCRIBE_FIGURES` is off, so you
can build the figure inventory on CPU and describe them in a later session.

Resume is keyed on `(source_pdf, absolute book page)` for pages and on `figure_id` for
figures, so an interrupted session, a retried error, and a re-run all converge without
duplicating final records.


In [ ]:
CHANDRA_BLOCKS_MAP = load_chandra_blocks(CHANDRA_PATH)
ocr_latest = load_latest(PAGE_OCR_JSONL) if RESUME else {}
figure_latest = load_latest(FIGURES_JSONL, key_fields=("figure_id",)) if RESUME else {}
processed = 0
run_started = time.perf_counter()

for pdf_index, pdf_file in enumerate(PDF_FILES, start=1):
    source_pdf = pdf_file.name
    doc = get_doc(pdf_file)
    offset = pdf_page_offset(pdf_file)
    first_book_page = offset + 1
    last_book_page = offset + doc.page_count

    lo = max(first_book_page, START_PAGE)
    hi = last_book_page if END_PAGE is None else min(last_book_page, END_PAGE)
    if PAGE_LIMIT is not None:
        hi = min(hi, lo + PAGE_LIMIT - 1)
    if lo > hi:
        print(f"\n=== {pdf_index}/{len(PDF_FILES)}: {source_pdf} | nothing in range, skipping ===")
        continue

    print(f"\n=== {pdf_index}/{len(PDF_FILES)}: {source_pdf} | book pages {lo}-{hi} ===")

    for book_page in range(lo, hi + 1):
        local_page = book_page - offset
        key = page_key(source_pdf, book_page)
        existing = ocr_latest.get(key)
        ocr_ok = bool(existing) and not existing.get("error")
        if existing and existing.get("error") and RETRY_ERRORS:
            ocr_ok = False

        print(f"{source_pdf} | book page {book_page}/{hi}")

        if not needs_vlm(source_pdf, book_page) and not ocr_ok:
            print("  routed away from the VLM by PAGE_QUALITY_CSV")
            continue

        if ocr_ok:
            ocr_result = existing
            print("  OCR: resumed")
        else:
            page = doc.load_page(local_page - 1)
            furniture = page_furniture_from_textlayer(page)
            image_path, image = page_image(pdf_file, local_page, book_page)
            try:
                ocr_result = vision_ocr_page_layout_aware(image, book_page, source_pdf, furniture, CHANDRA_BLOCKS_MAP)
                ocr_result["image_path"] = str(image_path)
                ocr_result["local_page"] = local_page
                print(f"  OCR: {ocr_result['page_type']} | words {ocr_result['word_count']} "
                      f"| printed {ocr_result['printed_page_vlm'] or ocr_result['printed_page_textlayer'] or '?'} "
                      f"| flags {','.join(ocr_result['quality_flags']) or '-'} "
                      f"| {ocr_result['usage']['completion_tokens']} tok "
                      f"| {ocr_result['usage']['elapsed_seconds']}s")
            except Exception as exc:
                ocr_result = {
                    "source_pdf": source_pdf, "page": book_page, "pdf_page": book_page,
                    "local_page": local_page, "page_type": "", "clean_text": "", "normalized_text": "",
                    "error": repr(exc), "raw_model_output": "", "usage": {},
                    "image_path": str(image_path),
                }
                print("  OCR ERROR:", repr(exc))
            append_jsonl(PAGE_OCR_JSONL, ocr_result)
            ocr_latest[key] = ocr_result
            processed += 1

        if ocr_result.get("error") or not EXTRACT_FIGURES:
            continue

        try:
            figures = detect_figures(pdf_file, local_page, book_page)
        except Exception as exc:
            print("  FIGURE DETECTION ERROR:", repr(exc))
            figures = []

        if not figures:
            continue
        print(f"  figures detected: {len(figures)}")

        page_context = (ocr_result.get("normalized_text") or "")[:1200]
        for figure in figures:
            fig_key = (figure["figure_id"],)
            existing_fig = figure_latest.get(fig_key)
            fig_ok = bool(existing_fig) and not existing_fig.get("error")
            if existing_fig and existing_fig.get("error") and RETRY_ERRORS:
                fig_ok = False

            if fig_ok:
                print(f"    {figure['figure_id']}: resumed")
                continue
            if not DESCRIBE_FIGURES:
                record = dict(figure)
                record["description"] = ""
                record["usage"] = {}
                append_jsonl(FIGURES_JSONL, record)
                figure_latest[fig_key] = record
                continue

            try:
                record = describe_figure(figure, page_context)
                print(f"    {record['figure_id']}: {record['figure_type']} "
                      f"| legend_agreement={record['legend_agreement']} "
                      f"| {record['usage']['elapsed_seconds']}s")
            except Exception as exc:
                record = dict(figure)
                record.update({"error": repr(exc), "description": "", "raw_model_output": "", "usage": {}})
                print(f"    {figure['figure_id']} ERROR:", repr(exc))
            append_jsonl(FIGURES_JSONL, record)
            figure_latest[fig_key] = record

        if processed and processed % SAVE_EVERY == 0:
            pages_df, figures_df, corpus_df = save_exports()
            elapsed = time.perf_counter() - run_started
            rate = elapsed / max(1, processed)
            print(f"  checkpoint: {len(pages_df)} pages, {len(figures_df)} figures, "
                  f"{len(corpus_df)} corpus rows | {rate:.1f}s/page this session")
            gc.collect()

pages_df, figures_df, corpus_df = save_exports(verbose=True)
elapsed = time.perf_counter() - run_started
print(f"\nRun complete in {elapsed / 60:.1f} min | {processed} new pages this session")
print(f"pages={len(pages_df)}  figures={len(figures_df)}  corpus rows={len(corpus_df)}")
if processed:
    remaining = max(0, TOTAL_BOOK_PAGES - len(pages_df))
    print(f"projected time for the remaining {remaining} pages: "
          f"{remaining * (elapsed / processed) / 3600:.1f} h")


## 12. Quality checks

Four things worth looking at before you trust the run, in order of how often they bite:

1. **Pages near the output token limit** — those transcriptions are truncated.
2. **Printed-page reconciliation** — how many pages had their number derived from the
   offset rather than read. A high count means the header parse is failing.
3. **`legend_agreement`** — every `contradicts` is a figure description that disagrees
   with what the book itself says the picture is. Read those by hand.
4. **Suspiciously short pages** — a prose page with 40 words is usually a silent refusal
   or a JSON truncation, not a short page.


In [ ]:
pages_df = pd.read_csv(PAGES_CSV) if PAGES_CSV.exists() else pd.DataFrame()
figures_df = pd.read_csv(FIGURES_CSV) if FIGURES_CSV.exists() else pd.DataFrame()
corpus_df = pd.read_csv(RAG_CORPUS_CSV) if RAG_CORPUS_CSV.exists() else pd.DataFrame()
token_df = pd.read_csv(TOKEN_LOG_CSV) if TOKEN_LOG_CSV.exists() else pd.DataFrame()
error_df = pd.read_csv(ERROR_LOG_CSV) if ERROR_LOG_CSV.exists() else pd.DataFrame()

print(f"pages={len(pages_df)}  figures={len(figures_df)}  corpus rows={len(corpus_df)}  errors={len(error_df)}")

if not pages_df.empty:
    print("\nTotal words transcribed:", int(pages_df["word_count"].sum()))
    print("\nPages by type:")
    display(pages_df["page_type"].value_counts())
    print("Printed page number resolved by:")
    display(pages_df["printed_page_source"].value_counts())
    print("Most common quality flags:")
    display(pages_df["quality_flags"].fillna("").str.split("|").explode().replace("", np.nan).dropna().value_counts().head(12))
    print("Suspiciously short prose pages (check for truncation or refusal):")
    display(pages_df[(pages_df["page_type"].isin(["prose", "mixed"])) & (pages_df["word_count"] < 60)]
            [["pdf_page", "printed_page", "page_type", "word_count", "quality_flags"]].head(20))

if not token_df.empty:
    display(token_df.groupby("stage")[["prompt_tokens", "completion_tokens", "elapsed_seconds"]].describe().round(2))
    if "hit_token_limit" in token_df.columns:
        truncated = token_df[token_df["hit_token_limit"].fillna(False).astype(bool)]
        print(f"Rows that hit the token limit (transcription truncated): {len(truncated)}")
        display(truncated.head(20))

if not figures_df.empty:
    print("\nFigures by detector:")
    display(figures_df["detector"].value_counts())
    print("Figures by type:")
    display(figures_df["figure_type"].value_counts())
    print("Legend agreement - every 'contradicts' is a likely hallucination, read these:")
    display(figures_df["legend_agreement"].value_counts())
    display(figures_df[figures_df["legend_agreement"].isin(["contradicts", "partial"])]
            [["figure_id", "printed_page", "figure_label", "printed_legend", "description"]].head(15))
    print("Figures with no printed legend (retrieval rests on the description alone):")
    print(int((figures_df["printed_legend"].fillna("").str.strip() == "").sum()))

if not corpus_df.empty:
    print("\nCorpus rows by modality:")
    display(corpus_df["modality"].value_counts())
    display(corpus_df["char_count"].describe().round(1))

if not error_df.empty:
    display(error_df.head(30))


### 12.1 Eyeball a figure record end to end

The crop, the caption the book printed, and what the model said. If these three do not
line up, nothing downstream will.


In [ ]:
if not figures_df.empty:
    for _, row in figures_df.head(4).iterrows():
        print("=" * 78)
        print(f"{row['figure_id']} | printed page {row['printed_page']} | {row.get('figure_type', '')} "
              f"| agreement={row.get('legend_agreement', '')}")
        print(f"PRINTED LEGEND : {row.get('printed_legend', '')}")
        print(f"MODEL SAYS     : {row.get('description', '')}")
        path = Path(str(row.get("crop_path", "")))
        if path.exists():
            display(resize_max_side(Image.open(path), 420))
else:
    print("No figure records yet.")


## 13. Handoff to the vector database

Embed `retrieval_text`; keep everything else as payload. `modality` lets one index hold
both prose chunks and figure descriptions, so a question like *"what does Fig. 8 show?"*
retrieves the figure record directly instead of dragging in a page of surrounding prose.


In [ ]:
if not corpus_df.empty:
    embedding_texts = corpus_df["retrieval_text"].fillna("").tolist()
    payloads = corpus_df.drop(columns=["retrieval_text"]).to_dict("records")
    print("Embedding texts:", len(embedding_texts))
    print("  text chunks   :", int((corpus_df["modality"] == "text").sum()))
    print("  figure records:", int((corpus_df["modality"] == "figure").sum()))

    for modality in ("text", "figure"):
        subset = corpus_df[corpus_df["modality"] == modality]
        if subset.empty:
            continue
        print(f"\n--- example {modality} record ---")
        print(subset.iloc[0]["retrieval_text"][:900])
        print("citation:", subset.iloc[0].get("citation", ""))

    print("\nPayload keys:", sorted(payloads[0].keys()))
else:
    print("No corpus rows yet. Run section 11 first.")


## 14. Output files

| File | What it is |
|---|---|
| `pcma_page_ocr.jsonl` | One record per page. Faithful `clean_text`, `normalized_text`, both page numbers, headings, figure legends, quality flags, exact model output, usage. **Keep this.** It is the audit trail, and it is what you score Tesseract against later. |
| `pcma_figures.jsonl` | One record per figure: crop paths, printed legend, model description, visible labels, `legend_agreement`. Stored separately from body text. |
| `pcma_chunks.jsonl` | Deterministic text chunks with `pdf_page_start/end` and `printed_page_start/end`. |
| `pcma_rag_corpus.jsonl` / `.csv` | Text chunks + figure records, tagged `modality`, with a ready `citation` string. Embed `retrieval_text`. |
| `pcma_pages.csv` | Page-level table for the EDA notebook: page type, word count, quality flags, how each printed page number was resolved. |
| `pcma_figures.csv` | Flat figure table for the EDA notebook. |
| `pcma_token_log.csv` | Tokens, seconds, and the truncation flag per call. |
| `pcma_errors.csv` | Failures. Re-run with `RESUME=True` and `RETRY_ERRORS=True`. |
| `pcma_page_images/`, `pcma_figures/` | Rendered pages and figure crops. Ship the crops with the index; a citation to a figure should be able to show it. |

### Running it

1. `PAGE_LIMIT = 5`, run everything. Check section 6.1's crops and section 12's tables.
2. Set `RENDER_DPI` from the native DPI printed in section 4.
3. `PAGE_LIMIT = None`, set `START_PAGE`/`END_PAGE` to a slice that fits your session, run.
4. Next session, same settings with the next slice. Resume skips finished pages.

### One thing this notebook does not do

It does not tell you whether Qwen is *right*. It is one OCR system, not the referee — its
characteristic failure on an 1875 book is fluent, invisible normalisation, which scores
well against itself. The `[illegible]` markers, `uncertain_spans` and `quality_flags` here
tell you where it *admits* difficulty, and `legend_agreement` catches figure descriptions
that contradict the book. Everything else needs the hand-corrected gold pages and the
Tesseract comparison, which are the next piece of work.


## 15. Benchmark Evaluation — CER / WER against Ground Truth

Calculates CER, WER, and Word F1 over held-out ground truth pages (`labels.jsonl`)
and compares Qwen3.5 against Chandra pseudo-ground-truth and baseline OCR models.


In [ ]:
import csv
from jiwer import cer as compute_cer, wer as compute_wer

EVAL_OUT_DIR = WORK_DIR / "qwen_layout_bench"
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

gt_labels = {}
if LABELS_PATH.exists():
    with LABELS_PATH.open(encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            row = json.loads(line)
            gt_labels[row["page_id"]] = row["text"]
    print(f"Loaded {len(gt_labels)} ground-truth pages for evaluation: {sorted(gt_labels.keys())}")
else:
    print(f"[WARN] Ground truth labels.jsonl not found at {LABELS_PATH}")

if gt_labels:
    ocr_map = load_latest(PAGE_OCR_JSONL)
    transcript_map = {}
    for k, v in ocr_map.items():
        pid = f"p{int(v.get('pdf_page', 0)):04d}"
        transcript_map[pid] = v.get("clean_text", "")

    scored_pages = []
    print("\npage_id      CER      WER   Word-F1")
    print("-" * 45)
    for page_id, ref_text in sorted(gt_labels.items()):
        hyp_text = transcript_map.get(page_id, "")
        ref_norm = clean_string(ref_text)
        hyp_norm = clean_string(hyp_text)
        if not ref_norm:
            continue
        c = compute_cer(ref_norm, hyp_norm)
        w = compute_wer(ref_norm, hyp_norm)
        ref_words = set(ref_norm.lower().split())
        hyp_words = set(hyp_norm.lower().split())
        tp = len(ref_words & hyp_words)
        prec = tp / len(hyp_words) if hyp_words else 0.0
        rec = tp / len(ref_words) if ref_words else 0.0
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
        scored_pages.append({"page_id": page_id, "cer": round(c, 4), "wer": round(w, 4), "word_f1": round(f1, 4)})
        print(f"{page_id}   {c:.4f}   {w:.4f}   {f1:.4f}")

    if scored_pages:
        q_cer = float(np.mean([p["cer"] for p in scored_pages]))
        q_wer = float(np.mean([p["wer"] for p in scored_pages]))
        q_f1  = float(np.mean([p["word_f1"] for p in scored_pages]))
        print("-" * 45)
        print(f"Qwen3.5 MEAN : CER={q_cer:.4f}  WER={q_wer:.4f}  Word-F1={q_f1:.4f}")

        score_csv = EVAL_OUT_DIR / "heldout_scores.csv"
        with score_csv.open("w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(scored_pages[0]))
            writer.writeheader()
            writer.writerows(scored_pages)
        print(f"Scores saved to {score_csv}")

        # Score Chandra pseudo-GT for comparison
        chandra_scores = []
        for page_id, ref_text in sorted(gt_labels.items()):
            chandra_text = " ".join(b.get("content", "") for b in CHANDRA_BLOCKS_MAP.get(page_id, []))
            ref_norm = clean_string(ref_text)
            hyp_norm = clean_string(chandra_text)
            if not ref_norm or not hyp_norm: continue
            chandra_scores.append((compute_cer(ref_norm, hyp_norm), compute_wer(ref_norm, hyp_norm)))

        c_cer = float(np.mean([s[0] for s in chandra_scores])) if chandra_scores else float("nan")
        c_wer = float(np.mean([s[1] for s in chandra_scores])) if chandra_scores else float("nan")

        report_lines = [
            "# Qwen3.5 Vision OCR Layout-Aware Benchmark",
            f"- Qwen3.5 Mean CER    : {q_cer:.4f}",
            f"- Qwen3.5 Mean WER    : {q_wer:.4f}",
            f"- Qwen3.5 Mean Word F1: {q_f1:.4f}",
            f"- Chandra Mean CER   : {c_cer:.4f}",
            f"- Chandra Mean WER   : {c_wer:.4f}",
        ]
        report_path = EVAL_OUT_DIR / "report.md"
        report_path.write_text("\n".join(report_lines) + "\n")
        print(f"\nReport saved to {report_path}:\n")
        print("\n".join(report_lines))
